In [7]:
import os
#os.environ["HF_HOME"] = "/data/cat/ws/kafu622g-propaganda/huggingface"
os.environ["HF_HOME"] = "/home/thhaase/miniforge3/envs/Internship/huggingface"

#print(f"Huggingface cache set to: {os.environ["HF_HOME"]}")

In [8]:
import json
import re
import pandas as pd

from vllm import LLM, SamplingParams

os.chdir(os.path.expanduser("~/GoogleDrive/_1_Projects/0_Internship/analysis")) 
data_path = "/home/thhaase/Documents/synosys_internship"

[W106 20:16:55.260474978 OperatorEntry.cpp:218] Warning: Warning only once for all operators,  other operators may also be overridden.
  Overriding a previously registered kernel for the same operator and the same dispatch key
  operator: aten::_addmm_activation(Tensor self, Tensor mat1, Tensor mat2, *, Scalar beta=1, Scalar alpha=1, bool use_gelu=False) -> Tensor
    registered at /pytorch/build/aten/src/ATen/RegisterSchema.cpp:6
  dispatch key: AutocastCPU
  previous kernel: registered at /pytorch/aten/src/ATen/autocast_mode.cpp:327
       new kernel: registered at /opt/workspace/ipex-cpu-dev/csrc/cpu/autocast/autocast_mode.cpp:112 (function operator())


INFO 01-06 20:16:57 [__init__.py:235] Automatically detected platform cpu.


In [9]:
dd = pd.read_parquet(os.path.join(data_path, "dd.parquet"))

In [10]:
# read prompt file
with open("prompt8.md", "r", encoding="utf-8") as f:
  prompt_system = f.read()

# LLM CODING

## Process Posts with LLM

In [11]:
# SETUP
#MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
MODEL_NAME = "/home/thhaase/Documents/synosys_internship/huggingface/Qwen2.5-Coder-0.5B-Instruct"
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

llm_kwargs = dict(
    model = MODEL_NAME,
    enforce_eager = True  
)

sampling_params = SamplingParams(
    temperature=0.7,
    max_tokens=4048,
    #top_p=0.9
)

llm = LLM(**llm_kwargs)
tokenizer = llm.get_tokenizer()

temp_sample = dd.sample(n=10)[["id", "xid", "text"]].reset_index(drop=True)

# PROCESS TEXT
prompts_list = []
rows_list = []

for row in temp_sample.itertuples():

    conversation = [
        {"role": "system", "content": prompt_system},
        {"role": "user", "content": row.text}
    ]
    
    formatted_prompt = tokenizer.apply_chat_template(
        conversation, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    prompts_list.append(formatted_prompt)
    rows_list.append(row)


print(f"Starting inference on {len(prompts_list)} items...")
outputs = llm.generate(prompts_list, sampling_params)

`torch_dtype` is deprecated! Use `dtype` instead!


INFO 01-06 20:17:06 [config.py:1604] Using max model len 32768
WARNING 01-06 20:17:06 [logger.py:71] Environment variable VLLM_CPU_KVCACHE_SPACE (GiB) for CPU backend is not set, using 4 by default.
INFO 01-06 20:17:06 [config.py:2434] Chunked prefill is enabled with max_num_batched_tokens=4096.


[W106 20:17:21.650929914 OperatorEntry.cpp:218] Warning: Warning only once for all operators,  other operators may also be overridden.
  Overriding a previously registered kernel for the same operator and the same dispatch key
  operator: aten::_addmm_activation(Tensor self, Tensor mat1, Tensor mat2, *, Scalar beta=1, Scalar alpha=1, bool use_gelu=False) -> Tensor
    registered at /pytorch/build/aten/src/ATen/RegisterSchema.cpp:6
  dispatch key: AutocastCPU
  previous kernel: registered at /pytorch/aten/src/ATen/autocast_mode.cpp:327
       new kernel: registered at /opt/workspace/ipex-cpu-dev/csrc/cpu/autocast/autocast_mode.cpp:112 (function operator())


INFO 01-06 20:17:22 [__init__.py:235] Automatically detected platform cpu.
INFO 01-06 20:17:23 [core.py:572] Waiting for init message from front-end.
INFO 01-06 20:17:23 [core.py:71] Initializing a V1 LLM engine (v0.10.0) with config: model='Qwen/Qwen2.5-1.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-1.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=True, quantization=None, enforce_eager=True, kv_cache_dtype=auto,  device_config=cpu, decoding_config=DecodingConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_backend=''), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collec

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.34it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.34it/s]



INFO 01-06 20:18:52 [default_loader.py:262] Loading weights took 0.83 seconds
INFO 01-06 20:18:52 [kv_cache_utils.py:833] GPU KV cache size: 149,760 tokens
INFO 01-06 20:18:52 [kv_cache_utils.py:837] Maximum concurrency for 32,768 tokens per request: 4.57x
INFO 01-06 20:18:52 [cpu.py:100] Using Torch SDPA backend.
INFO 01-06 20:18:53 [cpu_model_runner.py:63] Warming up model for the compilation...
INFO 01-06 20:18:54 [cpu_model_runner.py:67] Warming up done.
INFO 01-06 20:18:54 [core.py:193] init engine (profile, create kv cache, warmup model) took 1.53 seconds
WARNING 01-06 20:18:55 [logger.py:71] Environment variable VLLM_CPU_KVCACHE_SPACE (GiB) for CPU backend is not set, using 4 by default.
Starting inference on 10 items...


Processed prompts: 100%|██████████| 10/10 [01:54<00:00, 11.45s/it, est. speed input: 209.97 toks/s, output: 23.74 toks/s]


## Process Results

In [ ]:
d_results = []

for row, output in zip(rows_list, outputs):
    answer = output.outputs[0].text
    
    # --- 1. Attempt Standard JSON Parsing ---
    try:
        # Isolate the JSON object (find first { and last })
        json_str = answer[answer.find("{"):answer.rfind("}") + 1]
        answer_data = json.loads(json_str)
        
    except (json.JSONDecodeError, ValueError, AttributeError):
        # --- 2. Regex Fallback (Fixes "Bad Quotes" inside text) ---
        # The model often breaks JSON by putting double quotes inside strings.
        # We manually extract the fields based on your specific keys.
        try:
            parsed = {}
            
            # A. Extract Text Fields
            # Captures text between: "key": " ...AND... ", (comma) or "} (end brace)
            str_keys = [
                "holistic_redescription", 
                "social_actors_analysis", 
                "populist_explanation", 
                "elitist_explanation", 
                "intensity_explanation"
            ]
            
            for key in str_keys:
                match = re.search(f'"{key}"\s*:\s*"(.*?)"\s*(?:,|}}\s*$)', answer, re.DOTALL)
                if match:
                    # Clean up: Replace internal double quotes with single quotes to be safe
                    parsed[key] = match.group(1).replace('"', "'")
                else:
                    parsed[key] = None

            # B. Extract Number Fields
            num_keys = ["populist_score", "elitist_score", "intensity_score"]
            for key in num_keys:
                match = re.search(f'"{key}"\s*:\s*(-?\d+)', answer)
                if match:
                    parsed[key] = int(match.group(1))
                else:
                    parsed[key] = None
            
            answer_data = parsed
            
        except Exception as regex_error:
            # If both fail, log it
            answer_data = {"error": f"Parse Failed: {regex_error}", "raw_output": answer}

    # --- 3. Append to Results ---
    d_results.append({
        "id": row.id, 
        "xid": row.xid, 
        "text": row.text,
        **answer_data
    })

# Convert to DataFrame
d = pd.DataFrame(d_results)

<positron-console-cell-8>:30: SyntaxWarning: invalid escape sequence '\s'
<positron-console-cell-8>:30: SyntaxWarning: invalid escape sequence '\s'
<positron-console-cell-8>:40: SyntaxWarning: invalid escape sequence '\s'


# Save Results

In [7]:
d.to_parquet(f'{data_path}/llm_coding.parquet')